# Análise Exploratória dos Dados: Otimização dos parâmetros de soldagem

#### Objetivo Geral
Analisar o comportamento estatístico e físico dos dados para estruturar o script de pré-processamento e o código de modelagem (Regressão Multi-Output em PyTorch com ajuste de hiperparâmetros via Optuna).

####  Roteiro da Análise
1. **Preparação e Unificação dos Dados:** Unificação dos dados por tipo de passe (Raiz, Enchimento e Acabamento).
2. **Diagnóstico e Tratamento dos Dados Ausentes:** Tratamento de valores nulos e correção de textos/unidades.
3. **Análise da Cardinalidade das variáveis:** 
3. **Alvos:** Análise da distribuição estatística e identificação de outliers operacionais.
4. **Física do Processo:** Cruzamento de variáveis (GMAW, SMAW, TIG) e estudo de correlações não-lineares.
5. **Decisões:** Geração do checklist com as regras de negócio para o código de treino.

In [1]:
#Bibliotecas Utilizadas

# Sistema e Ambiente
import os
from pathlib import Path
from dotenv import load_dotenv

# Manipulação e Análise de Dados
import numpy as np
import pandas as pd

# Visualização de Dados
import matplotlib.pyplot as plt
import seaborn as sns


### 1. Preparação e Unificação dos Dados

In [2]:
load_dotenv()
DATA_PATH = os.getenv("CAMINHO_DADOS")
df = pd.read_csv(DATA_PATH, sep=';')

In [3]:
# Trata os nulos do diâmetro e cria a classificação 
df['material1_diametro'] = df['material1_diametro'].fillna(0.0)
df['tipo_peca'] = np.where(df['material1_diametro'] > 0, 'tubo', 'chapa')

In [4]:
colunas_comuns = [
    # norma
    'normas_referencia',

    # material de base
    'material1_pnumber',
    'material1_especificacao',
    'material1_espessura',
    'material1_diametro',
    'tipo_peca',

    # junta
    'tipo_chanfro',
    'angulo',
    'nariz',
    'abertura_raiz',
    'cobre_junta',
    'goivagem',

    # posição
    'posicao_peca',
    'progressao',

    # temperatura
    'pre_aquecimento',
    'temperatura_interpasse'
]

In [5]:
passes = {
    'raiz': {
        'processo': 'raiz_processo',
        'classificacao': 'raiz_classificacao',
        'diametro': 'raiz_diametro',
        'corrente': 'raiz_tipo_corrente',
        'polaridade': 'raiz_polaridade',
        'limpeza': 'raiz_limpeza', 
        'voltagem': 'raiz_voltagem',
        'amperagem': 'raiz_amperagem',
        'velocidade_de_soldagem': 'raiz_vel_soldagem',
        'tipo_gas_tocha': 'tipo_gas_tocha_raiz',
        'vazao_gas_tocha': 'vazao_gas_tocha_raiz',
        'tipo_gas_purga': 'tipo_gas_purga_raiz',
        'vazao_gas_purga': 'vazao_gas_purga_raiz'
    },

    'enchimento': {
        'processo': 'enchimento_processo',
        'classificacao': 'enchimento_classificacao',
        'diametro': 'enchimento_diametro',
        'corrente': 'enchimento_tipo_corrente',
        'polaridade': 'enchimento_polaridade',
        'limpeza': 'enchimento_limpeza',
        'voltagem': 'enchimento_voltagem',
        'amperagem': 'enchimento_amperagem',
        'velocidade_de_soldagem': 'enchimento_vel_soldagem',
        'tipo_gas_tocha': 'tipo_gas_tocha_acabamento', # Herda do acabamento
        'vazao_gas_tocha': 'vazao_gas_tocha_acabamento', # Herda do acabamento
        'tipo_gas_purga': 'tipo_gas_purga_acabamento', # Herda do acabamento
        'vazao_gas_purga': 'vazao_gas_purga_acabamento' # Herda do acabamento
    },

    'acabamento': {
        'processo': 'acabamento_processo',
        'classificacao': 'acabamento_classificacao',
        'diametro': 'acabamento_diametro',
        'corrente': 'acabamento_tipo_corrente',
        'polaridade': 'acabamento_polaridade',
        'limpeza': 'acabamento_limpeza',
        'voltagem': 'acabamento_voltagem',
        'amperagem': 'acabamento_amperagem',    
        'velocidade_de_soldagem': 'acabamento_vel_soldagem',
        'tipo_gas_tocha': 'tipo_gas_tocha_acabamento',
        'vazao_gas_tocha': 'vazao_gas_tocha_acabamento',
        'tipo_gas_purga': 'tipo_gas_purga_acabamento',
        'vazao_gas_purga': 'vazao_gas_purga_acabamento'
    }
}

In [6]:
dfs = []

for passe, cols in passes.items():
    temp = df[colunas_comuns].copy()
    temp['passe'] = passe
    
    temp['processo'] = df[cols['processo']]
    temp['classificacao'] = df[cols['classificacao']]
    temp['diametro_arame'] = df[cols['diametro']]
    temp['tipo_corrente'] = df[cols['corrente']]
    temp['polaridade'] = df[cols['polaridade']]
    temp['limpeza'] = df[cols['limpeza']]
    temp['voltagem'] = df[cols['voltagem']]
    temp['amperagem'] = df[cols['amperagem']]
    temp['velocidade_de_soldagem'] = df[cols['velocidade_de_soldagem']]
    temp['tipo_gas_tocha'] = df[cols['tipo_gas_tocha']]
    temp['vazao_gas_tocha'] = df[cols['vazao_gas_tocha']]
    temp['tipo_gas_purga'] = df[cols['tipo_gas_purga']] 
    temp['vazao_gas_purga'] = df[cols['vazao_gas_purga']] 
    
    dfs.append(temp)

dados = pd.concat(dfs, ignore_index=True)

In [7]:
dados = dados.dropna(subset=['processo'])

### 2. Diagnóstico e Tratamento dos Dados Ausentes

In [8]:
dados.isna().sum().sort_values(ascending=False)

goivagem                   752
vazao_gas_purga            455
cobre_junta                419
vazao_gas_tocha            184
velocidade_de_soldagem      84
pre_aquecimento             45
temperatura_interpasse      45
voltagem                    23
nariz                       17
amperagem                   17
diametro_arame              11
angulo                       9
material1_pnumber            9
limpeza                      9
abertura_raiz                8
posicao_peca                 3
material1_diametro           0
tipo_chanfro                 0
material1_especificacao      0
material1_espessura          0
tipo_peca                    0
normas_referencia            0
progressao                   0
polaridade                   0
classificacao                0
processo                     0
passe                        0
tipo_corrente                0
tipo_gas_tocha               0
tipo_gas_purga               0
dtype: int64

In [9]:
dados['goivagem'].unique()

array([nan, True, False], dtype=object)

In [10]:
dados['cobre_junta'].unique()

array([nan, True, False], dtype=object)

In [11]:
col = ['goivagem', 'cobre_junta']
for c in col:
    dados[c] = dados[c].fillna('Informação Desconhecida')

In [12]:
dados['vazao_gas_purga'].unique()

array([nan, '9.0', '10.0', '25', '18', '8/12', '15', '12', '10/12', '10',
       '20/22', '13', '20', '16', '9', '23', '8/10', '22', '30',
       '10 / 15', '10 / 20', '11/15', '20 / 25', '10/15', '12/20',
       '15/25', '14', '20/25', '15/20', '16 mm'], dtype=object)

In [13]:
dados['vazao_gas_tocha'].unique()

array([nan, '12', '16', '15', '20', '25', '30', '18', '10/10.5', '11',
       '10', '11/13', '15/18', '12/20', '15/20', '14', '10/12', '13/15',
       '20/25', '7.5', '23', '16/30', '22', '9/15', '10/19', '9/16', '19',
       '16/25', '18/20', '19/21', '22/24', '18/22', '8/12', '9.5',
       '13/14', '13-15', '13', '8/10', '18 /20', '7', '8 / 12', '14 / 16',
       '11/15', '15 / 20', '15/25', '17', '17/18', '10/15', '7/12',
       '8/15', '15.0', '20.0', '25.0', '18.0', '12.0', '24', '14/24'],
      dtype=object)

In [14]:
import re 
def limpar_vazao(val):
    if pd.isna(val):
        return 0.0
    val_str = str(val).lower().strip()
    val_str = val_str.replace('dez', '12')
    val_str = val_str.replace(' a ', '/').replace('-', '/')
    numeros = re.findall(r'\d+\.\d+|\d+', val_str)
    if not numeros:
        return 0.0
    valores_float = [float(n) for n in numeros]
    return sum(valores_float) / len(valores_float)

cols = ['vazao_gas_purga', 'vazao_gas_tocha']
for c in cols:
   dados[c] = dados[c].apply(limpar_vazao)

In [15]:
dados['pre_aquecimento'] = dados['pre_aquecimento'].fillna(25.0)
dados['temperatura_interpasse'] = dados['temperatura_interpasse'].fillna(25.0)

In [16]:
col2 = ['nariz', 'abertura_raiz', 'angulo']
for col in col2:
    dados[col] = dados[col].fillna(0.0)

In [17]:
dados['diametro_arame'] = dados['diametro_arame'].fillna(0.0)

In [18]:
dados['material1_pnumber'] = dados['material1_pnumber'].fillna('Não Informado').astype(str)
dados['posicao_peca'] = dados['posicao_peca'].fillna('Não Informado').astype(str)

In [19]:
dados = dados.dropna(subset=['voltagem', 'amperagem', 'velocidade_de_soldagem'])

In [20]:
dados['material1_espessura'] = dados['material1_espessura'].fillna(dados['material1_espessura'].median())

Nesta seção, realizamos o tratamento dos valores nulos (NaN) com base em regras de negócio e na física do processo de soldagem, preparando a base para a rede neural.As seguintes ações foram tomadas:

* Variáveis categóricas (goivagem, cobre_junta): Preenchidas com o texto 'Informação Desconhecida', já que a ausência de registro indica que o recurso não foi utilizado.
* Vazões de gás (vazao_gas_purga, vazao_gas_tocha): Tratadas com uma função baseada em Expressões Regulares (re) para extrair e calcular a média de valores textuais complexos (como faixas e intervalos). Valores nulos foram zerados.
* Parâmetros térmicos (pre_aquecimento, temperatura_interpasse): Substituídos por 25.0, representando a temperatura ambiente em graus Celsius para os casos onde não houve controle térmico.
* Geometria da junta e consumíveis (nariz, abertura_raiz, angulo, diametro_arame): Preenchidos com 0.0, indicando a ausência física dessas características no experimento.
* Identificadores textuais (material1_pnumber, posicao_peca): Substituídos por 'Não Informado' e convertidos explicitamente para string.
* Variáveis alvo / Targets (voltagem, amperagem, velocidade_de_soldagem): Remoção direta das linhas vazias via .dropna(), garantindo que o modelo seja treinado apenas com amostras que possuem o todos os valores.
* Espessura do material (material1_espessura): Preenchida com a mediana do dataset para manter o valor alinhado aos padrões comerciais de bitolas de chapas e tubos.

### 3. Tratamento de Dados Textuais

In [21]:
for c in dados.nunique().sort_values(ascending=False).index:
    print(f'{c}: {dados[c].nunique()} valores únicos')

velocidade_de_soldagem: 360 valores únicos
amperagem: 229 valores únicos
voltagem: 122 valores únicos
classificacao: 82 valores únicos
material1_espessura: 82 valores únicos
material1_especificacao: 73 valores únicos
normas_referencia: 67 valores únicos
temperatura_interpasse: 60 valores únicos
pre_aquecimento: 51 valores únicos
material1_diametro: 35 valores únicos
vazao_gas_tocha: 29 valores únicos
limpeza: 26 valores únicos
diametro_arame: 25 valores únicos
abertura_raiz: 24 valores únicos
vazao_gas_purga: 19 valores únicos
angulo: 15 valores únicos
nariz: 15 valores únicos
material1_pnumber: 14 valores únicos
posicao_peca: 10 valores únicos
tipo_chanfro: 6 valores únicos
tipo_gas_tocha: 6 valores únicos
progressao: 6 valores únicos
processo: 6 valores únicos
polaridade: 4 valores únicos
tipo_gas_purga: 4 valores únicos
goivagem: 3 valores únicos
passe: 3 valores únicos
cobre_junta: 3 valores únicos
tipo_peca: 2 valores únicos
tipo_corrente: 2 valores únicos


In [22]:
dados['normas_referencia'].unique()

array(['AWS D1.1 e N 133 h', 'ASME Seção IX Ed.2007',
       'ASME Seção IX Ed.2007 Add. 2008', 'ASME Seção IX Ed.2010',
       'ASME Seção IX - Ed. 1998/ N133G', 'ASME Seção IX Ed.1998/ N133G',
       'ASME Seção IX Ed. 1999/ N133G', 'ASME Seção IX-Ed. 1999/ N133G',
       'ASME Seção IX/1998 Add. 1999',
       'ASME Seção IX / Ed. 1998 - Add. 1999', 'ASME Seção IX Ed. 2000',
       'ASME Seção IX - 2001', 'ASME Seção IX Ed. 1995/ N133G',
       'ASME Seção IX - Ed. 1995', 'ASME Seção IX Ed. 1995',
       'ASME Seção IX Add. 97', 'AWS D1.1 Edição 2010', 'ASME IX 2013',
       'ASME IX 2010', 'ABS 2010', 'ASME IX 2007', 'ASME IX 2004',
       'ASME IX', 'AWS D1.1 2010', 'AWS D1.1 2008', 'AWS D1.1 2015',
       'AWS D1.1 2016', 'ABS 2015', 'AWS D1.1 2010 / ABS 2010',
       'ASME IX 2014', 'ASME Seção IX Ed. 1998',
       'ASME Seção IX Ed. 1998/ N133G', 'ASME Seção IX Ed. 1998/ N133J',
       'ASME Seção IX', 'ASME Seção IX Ed. 2004 Add.2006',
       'ASME Seção IX Ed. 1997', 'ASME Seç

In [23]:
def limpar_normas_referencia(texto):
    if pd.isna(texto):
        return 'Não Informado'
    texto = str(texto).lower()
    if re.search(r'\baws\b', texto):
        return 'AWS'
    if re.search(r'\basme\b.*\bix\b', texto) or re.search(r'\bix\b.*\basme\b', texto):
        return 'ASME IX'
    if re.search(r'\babs\b', texto):
        return 'ABS'
    if re.search(r'\bastm\b', texto):
        return 'ASTM'
    return 'Outros'

dados['normas_referencia'] = dados['normas_referencia'].apply(limpar_normas_referencia)

In [24]:
dados['normas_referencia'].unique()

array(['AWS', 'ASME IX', 'ABS', 'ASTM'], dtype=object)

In [25]:
dados['limpeza'].unique()

array(['ESMER.', 'ESMER./ESCOV', 'ESCOVAMENTO/ESMERILHAMENTO', 'NAN',
       'ESMER', 'INICIAL: AO METAL BRILHANTE',
       'DISCO ABRASIVO/ESCOVAMENTO', 'ESCOVAMENTO', 'DISCO/ESCOVAMENTO',
       'ESMERILHAMENTO', 'ESMERILHAMENTO/LIXAMENTO', 'ESCOV./ESMER.',
       'ESCOV./ESMERIL.', nan, 'ESMERIL.', 'LIXAMENTO', 'OBS1', 'ESCOV.',
       'ESCOV/ESMER', 'ESMER/ESCOV', 'AO METAL BRILHANTE',
       'ESCOV./SOLVENTE', 'DISCO E ESCOVA DE INOX',
       'ESMERILHAMENTO/ESCOVAMENTO', 'DISCO ABRASIVO + ESCOVA AÇO',
       'ESMER./ESCOV.', 'ESCOV'], dtype=object)

In [26]:
mapeamento_limpeza = {
    'ESMER.': 'ESMERILHAMENTO',
    'ESMER': 'ESMERILHAMENTO',
    'ESMERILHAMENTO': 'ESMERILHAMENTO',
    'ESMERIL.': 'ESMERILHAMENTO',
    'LIXAMENTO': 'LIXAMENTO',
    'ESCOVAMENTO': 'ESCOVAMENTO',
    'ESCOV.': 'ESCOVAMENTO',
    'ESCOV': 'ESCOVAMENTO',
    'ESMER./ESCOV': 'ESMERILHAMENTO + ESCOVAMENTO',
    'ESCOVAMENTO/ESMERILHAMENTO': 'ESMERILHAMENTO + ESCOVAMENTO',
    'DISCO ABRASIVO/ESCOVAMENTO': 'ESMERILHAMENTO + ESCOVAMENTO',
    'DISCO/ESCOVAMENTO': 'ESMERILHAMENTO + ESCOVAMENTO',
    'ESMERILHAMENTO/LIXAMENTO': 'ESMERILHAMENTO + LIXAMENTO',
    'ESCOV./ESMER.': 'ESMERILHAMENTO + ESCOVAMENTO',
    'ESCOV./ESMERIL.': 'ESMERILHAMENTO + ESCOVAMENTO',
    'ESCOV/ESMER': 'ESMERILHAMENTO + ESCOVAMENTO',
    'ESMER/ESCOV': 'ESMERILHAMENTO + ESCOVAMENTO',
    'ESMERILHAMENTO/ESCOVAMENTO': 'ESMERILHAMENTO + ESCOVAMENTO',
    'DISCO ABRASIVO + ESCOVA AÇO': 'ESMERILHAMENTO + ESCOVAMENTO',
    'DISCO E ESCOVA DE INOX': 'ESMERILHAMENTO + ESCOVAMENTO',
    'ESMER./ESCOV.': 'ESMERILHAMENTO + ESCOVAMENTO',
    'ESCOV./SOLVENTE': 'ESCOVAMENTO + SOLVENTE',
    'INICIAL: AO METAL BRILHANTE': 'METAL BRILHANTE',
    'AO METAL BRILHANTE': 'METAL BRILHANTE',
    'NAN': np.nan,
    'nan': np.nan,
    'OBS1': np.nan,
    'N/A': np.nan
}

dados['limpeza'] = dados['limpeza'].astype(str).str.strip().map(mapeamento_limpeza)
dados = dados.dropna(subset=['limpeza'])

In [27]:
dados['limpeza'].unique()

array(['ESMERILHAMENTO', 'ESMERILHAMENTO + ESCOVAMENTO',
       'METAL BRILHANTE', 'ESCOVAMENTO', 'ESMERILHAMENTO + LIXAMENTO',
       'LIXAMENTO', 'ESCOVAMENTO + SOLVENTE'], dtype=object)

In [28]:
dados['processo'].unique()

array(['ER', 'TIG', 'FCAW', 'MIG', 'SAW', 'TIG/ER'], dtype=object)

In [29]:
# Remove linhas em que o processo é 'SAW' ou 'TIG/ER'
dados = dados[~dados['processo'].isin(['SAW'])]
dados = dados[~dados['processo'].str.contains('TIG/ER', case=False, na=False)]

In [30]:
dados['progressao'].unique()

array(['ASCENDENTE', 'NAN', 'DESCENDENTE/ASCENDENTE',
       'ASCENDENTE / ASCENDENTE + DESCENDENTE'], dtype=object)

In [31]:
nome_coluna = 'progressao'

dados[nome_coluna] = dados[nome_coluna].astype(str).str.strip().str.upper()

dados[nome_coluna] = dados[nome_coluna].replace({'NAN': np.nan, 'NAN ': np.nan, 'nan': np.nan})

def ajustar_progressao_por_passe(linha):
    prog = linha[nome_coluna]
    passe = linha['passe']
    
    if pd.isna(prog):
        return np.nan    
    if '/' in prog:
        partes = prog.split('/') 
        if passe == 'raiz':
            return 'DESCENDENTE' if 'DESCENDENTE' in partes[0] else 'ASCENDENTE'
        else:
            return 'ASCENDENTE' if 'ASCENDENTE' in partes[1] else 'DESCENDENTE'       
    if '+' in prog and passe == 'acabamento':
        return 'DESCENDENTE' 
    return prog

dados[nome_coluna] = dados.apply(ajustar_progressao_por_passe, axis=1)
dados = dados.dropna(subset=[nome_coluna])

In [32]:
dados['progressao'].unique()

array(['ASCENDENTE', 'DESCENDENTE'], dtype=object)

In [33]:
dados['polaridade'].unique()

array(['INVERSA', 'DIRETA', 'NAN'], dtype=object)

In [39]:
dados['polaridade'] = dados['polaridade'].replace('NAN', 'Informação Desconhecida')

In [40]:
dados['polaridade'].unique()

array(['INVERSA', 'DIRETA', 'Informação Desconhecida'], dtype=object)

Nesta seção, realizamos a limpeza textual das variáveis categóricas com base em regras de negócio e na física do processo de soldagem, preparando a base para a rede neural. As seguintes ações foram tomadas:
* Limpeza Textual e Filtro de Processos: Conversão de strings para maiúsculas, remoção de espaços extras e exclusão dos processos SAW e TIG/ER para delimitar o escopo.
* Padronização de Normas via Regex: Agrupamento de textos variados nas categorias oficiais AWS, ASME IX, ABS, ASTM ou Outros.
* Unificação de Sinônimos de Limpeza: Eliminação de abreviações e mapeamento de termos equivalentes, como o uso de disco abrasivo para o processo de esmerilhamento.
* Separação Dinâmica da Progressão: Quebra de textos combinados avaliando o tipo de passe da linha para definir se o sentido real daquela etapa foi puramente ascendente ou descendente.
* Tratamento de Nulos: Eliminação de ruídos textuais como OBS1 e NAN em colunas críticas e preenchimento de vazios em polaridade como Informação Desconhecida.